# Module 9 · 圖像 1：經典影像特徵（快速帶過）

> **本節定位｜快速帶過（經典基礎）**
> 色彩直方圖、HOG 是深度學習普及前（~2005–2015）的**手工影像特徵**。
> 它們仍有助於建立「如何把像素變成特徵」的直覺，但 **2026 主流是用
> 預訓練視覺模型（ViT/CLIP）自動學特徵**（下一節起）。本節快速體驗即可。

## 學習目標
- 理解影像在電腦中的本質：**像素張量**。
- 用 **色彩直方圖** 與 **HOG** 兩種經典手工特徵，把影像變成固定長度向量。
- 認識手工特徵的限制（不具語意、對視角/光照敏感），理解為何被學習式特徵取代。

In [1]:
import numpy as np
from PIL import Image

# 自製一張合成影像（無需下載任何資料集）：左半藍、右半橘，加一點雜訊
H, W = 128, 128
img = np.zeros((H, W, 3), dtype=np.uint8)
img[:, :W//2] = [40, 80, 200]    # 左半偏藍
img[:, W//2:] = [230, 140, 30]   # 右半偏橘
img = (img + np.random.RandomState(0).randint(0, 20, img.shape)).clip(0, 255).astype(np.uint8)
pil_img = Image.fromarray(img)
print(f"影像張量形狀 = {img.shape}  (H, W, C) ← 注意這是 numpy/PIL 慣例")
print(f"像素值域 = [{img.min()}, {img.max()}]  dtype = {img.dtype}")

影像張量形狀 = (128, 128, 3)  (H, W, C) ← 注意這是 numpy/PIL 慣例
像素值域 = [30, 249]  dtype = uint8


## 1. 色彩直方圖 (Color Histogram)

統計每個顏色通道的像素強度分佈，串接成一個向量。**對平移/旋轉穩健，但完全丟失空間結構**。

In [2]:
def color_histogram(arr, bins=16):
    feats = []
    for ch in range(3):                       # R, G, B 三通道
        hist, _ = np.histogram(arr[:, :, ch], bins=bins, range=(0, 255))
        feats.append(hist)
    return np.concatenate(feats).astype(float)

hist_vec = color_histogram(img, bins=16)
print(f"色彩直方圖特徵維度 = {hist_vec.shape}  (3 通道 × 16 bins = 48)")
print("→ 不論影像多大，都壓成固定長度向量；但『藍在左、橘在右』這個空間資訊完全消失。")

色彩直方圖特徵維度 = (48,)  (3 通道 × 16 bins = 48)
→ 不論影像多大，都壓成固定長度向量；但『藍在左、橘在右』這個空間資訊完全消失。


## 2. HOG (Histogram of Oriented Gradients)

統計局部區域的「梯度方向分佈」，能捕捉**邊緣/形狀**，曾是行人偵測的經典特徵。
需要 `classical` extra：`uv sync --extra classical`。

In [3]:
try:
    from skimage.feature import hog
    from skimage.color import rgb2gray
    gray = rgb2gray(img)
    hog_vec = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                  cells_per_block=(2, 2), feature_vector=True)
    print(f"HOG 特徵維度 = {hog_vec.shape}")
    print("→ 捕捉了邊緣方向（左右交界的垂直邊緣會有明顯響應）。")
except Exception as e:
    print("（未安裝 scikit-image，略過實際計算）", e)
    print("概念：把影像切成小格，統計每格的梯度方向直方圖，串接成特徵；擅長描述形狀/邊緣。")

（未安裝 scikit-image，略過實際計算） No module named 'skimage'
概念：把影像切成小格，統計每格的梯度方向直方圖，串接成特徵；擅長描述形狀/邊緣。


## 3. 手工特徵的限制 → 為何 2026 改用學習式特徵

| 面向 | 手工特徵（本節） | 學習式特徵（下一節 ViT/CLIP） |
|:--|:--|:--|
| 特徵來源 | 人類設計規則（顏色統計、梯度） | 模型在海量影像上**自己學** |
| 語意 | 無（只看低階統計） | 有（能分辨貓/狗/物件/場景） |
| 穩健性 | 對視角、光照、形變敏感 | 大幅較穩健 |
| 通用性 | 換任務常要重新設計 | 一個預訓練模型可遷移到多任務 |

**結論**：手工特徵幫助理解「像素→向量」，但要餵 2026 的模型，
我們需要把影像**正確地變成張量**（下一節），再交給預訓練視覺模型抽特徵。

---
### 小結
- 影像 = 像素張量 `(H, W, C)`。
- 色彩直方圖：固定長度、丟空間資訊；HOG：捕捉邊緣形狀。
- 兩者皆無語意、對變形敏感 → 被 ViT/CLIP 等學習式特徵取代。